# Additive Technique Notebook 2: Multimodal RAG for Startup Intelligence

This notebook adds **Multimodal RAG** over SEC/company intelligence sources by ingesting:
- text chunks (existing)
- financial/disclosure tables (new)
- figure/caption/alt-text channels (new)

Status in this phase: implementation complete, execution deferred; outputs are placeholders.



## 1. Why Multimodal RAG Exists

SEC filing intelligence is not purely narrative text. Critical signals can appear in:
- tabular financial disclosures
- visual/diagram captions
- image alt text in filing HTML

Multimodal RAG broadens evidence coverage beyond plain paragraphs.



## 2. Architecture Diagram

```mermaid
flowchart LR
    A[Filing Sources] --> B[Text Chunk Pipeline]
    A --> C[Table Extractor]
    A --> D[Figure Text Extractor]
    B --> E[Unified Multimodal Units]
    C --> E
    D --> E
    E --> F[Multimodal Retriever
Dense+Sparse + modality weights]
    F --> G[Grounded Generation
granite4.1:8b]
    G --> H[Guardian Evaluation
granite4.1:8b]
```



## 3. Workflow Diagram

```mermaid
sequenceDiagram
    participant I as Ingestion
    participant M as Multimodal Builder
    participant R as Multimodal Retriever
    participant G as Generator
    participant J as Guardian Judge

    I->>M: Filings + HTML source map
    M->>M: Extract table units + figure-text units
    M-->>R: Unified multimodal units
    R->>R: Dense+sparse retrieval with modality weighting
    R-->>G: Top-K multimodal contexts
    G-->>J: Answer + citations
    J-->>G: Judge metrics + RAG quality metrics
```



## 4. Tradeoffs and Design Choices

Chosen approach in this project:
- Use **table text canonicalization** and **figure text channels** from filing HTML.
- This v1 notebook intentionally avoids OCR/vision model dependencies; see notebook 04 for the OCR + vision implementation.

Why this choice:
- Works with the mandated local model stack.
- Keeps implementation reproducible and execution-friendly on Ubuntu.

Limitations:
- Raw image semantics are only captured if represented in alt/caption text.
- Complex chart understanding is covered in the additive OCR/vision workflow in notebook 04.



In [1]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SETTINGS
from src.extensions.multimodal import build_multimodal_units_from_html_map
from src.extensions.multimodal_retriever import MultimodalRetriever
from src.extensions.benchmark import evaluate_retriever_end_to_end
from src.eval_queries import load_eval_queries
from src.judge import judge_answer

RUN_EXECUTION = False

print("Embedding model:", SETTINGS.embed_model)
print("Generator model:", SETTINGS.generator_model)
print("Judge model (extension):", "granite4.1:8b")

Embedding model: qwen3-embedding:4b
Generator model: granite4.1:8b
Judge model (extension): granite4.1:8b


## 5. Build Multimodal Units

Expected input for later execution:
- `filings`: normalized filings from existing ingestion
- `filing_html_sources`: map `filing_id -> html string or local html file path`



In [2]:
if RUN_EXECUTION:
    # Example structure for execution phase:
    # filing_html_sources = {
    #   "AMD_10-K_2024_xxxxx": "/path/to/amd_10k.html",
    #   "ABT_10-K_2024_xxxxx": "<html>...</html>",
    # }
    multimodal_units = build_multimodal_units_from_html_map(
        filings=filings,
        filing_html_sources=filing_html_sources,
    )
    print("multimodal units:", len(multimodal_units))
else:
    print("Placeholder mode: multimodal unit build not executed.")


Placeholder mode: multimodal unit build not executed.


## 6. Multimodal Retrieval and Generation Flow



In [3]:
if RUN_EXECUTION:
    multimodal_retriever = MultimodalRetriever(multimodal_units)
    query = "What financial and strategic signals suggest margin pressure across companies?"
    hits = multimodal_retriever.retrieve(query, k=8)
    for i, hit in enumerate(hits, 1):
        print(i, hit.ticker, hit.metadata.get("modality"), round(hit.score, 4))
else:
    print("Placeholder mode: multimodal retrieval is not executed.")


Placeholder mode: multimodal retrieval is not executed.


## 7. Evaluation Methodology for Multimodal RAG

### Retrieval Metrics
- Precision@K
- Recall@K
- F1 Score
- MRR
- NDCG

### Generation Metrics
- Exact Match
- BLEU
- ROUGE-L
- METEOR
- BERTScore

### RAG Metrics
- Faithfulness
- Context Precision
- Context Recall
- Answer Relevancy

### LLM-as-a-Judge
- Model: `granite4.1:8b`



In [4]:
if RUN_EXECUTION:
    multimodal_report = evaluate_retriever_end_to_end(
        retriever=multimodal_retriever,
        queries=load_eval_queries(),
        metadata=[
            {
                "chunk_id": u.unit_id,
                "filing_id": u.filing_id,
                "ticker": u.ticker,
                "company_name": u.company_name,
                "section": u.section,
                "text": u.text,
            }
            for u in multimodal_units
        ],
        k=SETTINGS.default_top_k,
        judge_fn=judge_answer,
        judge_model="granite4.1:8b",
    )

    Path("artifacts/eval/multimodal_rag_full_metrics.json").write_text(
        json.dumps(multimodal_report.to_dict(), indent=2), encoding="utf-8"
    )
else:
    print("Placeholder mode: multimodal benchmark pipeline not executed.")


Placeholder mode: multimodal benchmark pipeline not executed.


## 8. Placeholder Outputs (Current Phase)

- `artifacts/eval/multimodal_rag_retrieval_metrics_placeholder.json`
- `artifacts/eval/multimodal_rag_generation_metrics_placeholder.json`
- `artifacts/eval/multimodal_rag_rag_metrics_placeholder.json`
- `artifacts/eval/multimodal_rag_judge_placeholder.json`



In [5]:
placeholder_files = [
    "artifacts/eval/multimodal_rag_retrieval_metrics_placeholder.json",
    "artifacts/eval/multimodal_rag_generation_metrics_placeholder.json",
    "artifacts/eval/multimodal_rag_rag_metrics_placeholder.json",
    "artifacts/eval/multimodal_rag_judge_placeholder.json",
]
for f in placeholder_files:
    payload = json.loads(Path(f).read_text(encoding="utf-8"))
    print(f, "->", payload["status"])


artifacts/eval/multimodal_rag_retrieval_metrics_placeholder.json -> executed
artifacts/eval/multimodal_rag_generation_metrics_placeholder.json -> executed
artifacts/eval/multimodal_rag_rag_metrics_placeholder.json -> executed
artifacts/eval/multimodal_rag_judge_placeholder.json -> executed


## 9. Limitations and Next Steps

Current limitations:
- Image understanding is limited to textual channels available in filing HTML.
- Modality weights are static and uncalibrated against real run metrics.

Execution-phase next steps:
1. Run end-to-end multimodal benchmark.
2. Compare text-only vs multimodal uplift across query types.
3. Add modality ablation and error analysis by question class.



## 10. What Is This Technique? (Definition and Motivation)

### Definition and Core Concepts
Multimodal RAG extends retrievable context beyond plain paragraphs by adding table and figure-text channels from filing documents.

### Why This Technique Was Developed
Traditional retrieval pipelines can underperform when one retrieval signal dominates (semantic-only or lexical-only or modality-only). This technique broadens evidence access while preserving grounding.

### What Limitations of Traditional RAG It Solves
- weak lexical recall for ticker/section-specific language
- weak semantic coverage for exact terminology
- missed evidence when relevant information is represented in alternative structures (for example tables/visual channels)





## 11. Architecture, Components, and Real-World Usage

### Component-by-Component Breakdown
1. Input preparation: filing-derived evidence units and metadata.
2. Retrieval orchestration: combine relevant retrieval signals for this technique.
3. Context selection: rank and keep top-K grounded contexts.
4. Generation: produce answer constrained by retrieved evidence.
5. Evaluation: compute retrieval, generation, RAG, and judge metrics.

### When To Use It in Real-World Systems
- high-stakes domains where evidence completeness matters
- corpora with both narrative and structured information
- systems requiring transparent, auditable retrieval behavior

### Advantages
- higher recall coverage across evidence types
- better robustness for diverse query styles
- clearer error analysis via separated metrics

### Disadvantages
- higher implementation complexity
- possible latency increase due to extra retrieval/model steps
- requires tighter evaluation discipline




## 12. Comparisons and Project Design Decisions

### Comparison Against Standard RAG and Other Implemented Variants
- vs Standard RAG: recovers evidence from structured/non-paragraph channels.
- vs Hybrid text-only: broader evidence coverage, potentially higher latency.
- vs OCR/Vision v2 variant: lighter and simpler, but less capable on scanned/non-HTML visuals.

### Implementation Details and Design Decisions in This Project
- additive-only design: existing working flows are preserved
- local model stack and strict dataset policy are maintained
- placeholder-first execution policy remains active until explicit run




## 13. Post-Run Analysis Template (Populate After Execution)

After real execution, replace placeholders with measured values and evidence.

### Actual Outputs and Metric Interpretation
- retrieval metrics (Precision@K, Recall@K, F1, MRR, NDCG): interpret query-level wins/failures.
- generation metrics (EM, BLEU, ROUGE-L, METEOR, BERTScore): explain answer-quality changes.
- RAG metrics (Faithfulness, Context Precision/Recall, Answer Relevancy): identify grounding improvements.
- judge metrics (correctness, relevance, completeness, groundedness, hallucination risk): summarize quality/risk profile.

### Retrieval Quality, Latency, and Generation Quality
- explain where latency increased/decreased and why.
- explain which query families improved and which did not.
- link improvements to concrete retrieved evidence patterns.

### Observations, Lessons Learned, and Practical Takeaways
- what worked reliably
- what failed and likely root causes
- what to adjust in next iteration (chunking, weighting, prompting, ranking, tool routing)

### Final Conclusion (Measured)
Summarize effectiveness using real measured results and explicit tradeoffs.




## Real Run Snapshot

Loads the latest real artifact for multimodal (HTML/table-aware) retrieval + generation.

In [6]:
# REAL_RUN_SNAPSHOT_MULTIMODAL_V1
from pathlib import Path
import json

path = Path('artifacts/eval/multimodal_rag_full_metrics.json')
if path.exists():
    payload = json.loads(path.read_text(encoding='utf-8'))
    retrieval = payload.get('retrieval_metrics', {})
    generation = payload.get('generation_metrics', {})
    rag = payload.get('rag_metrics', {})
    print('retrieval:', {k: retrieval.get(k) for k in ['precision_at_k', 'recall_at_k', 'f1_at_k', 'mrr', 'ndcg_at_k']})
    print('generation:', {k: generation.get(k) for k in ['em', 'bleu', 'rouge_l', 'meteor', 'bert_score_f1']})
    print('rag:', {k: rag.get(k) for k in ['faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']})
else:
    print('multimodal_rag_full_metrics.json not found. Execute scripts/run_full_real_project.py first.')


retrieval: {'precision_at_k': 0.0, 'recall_at_k': 0.0, 'f1_at_k': 0.0, 'mrr': 0.0, 'ndcg_at_k': 0.0}
generation: {'em': 0.0, 'bleu': 0.0054, 'rouge_l': 0.0826, 'meteor': 0.0763, 'bert_score_f1': -0.0037}
rag: {'faithfulness': 0.0, 'context_precision': 0.0, 'context_recall': 0.0, 'answer_relevancy': 0.0}
